In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Magsimula sa circuit cutting gamit ang wire cuts


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
qiskit-addon-cutting~=0.10.0
```
</details>
Ipinapakita ng gabay na ito ang isang gumaganang halimbawa ng wire cuts gamit ang `qiskit-addon-cutting` package. Sinasaklaw nito ang pag-reconstruct ng mga expectation value ng isang circuit na may pitong qubit gamit ang wire cutting.

Ang isang wire cut ay kinakatawan sa package na ito bilang isang two-qubit [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) na instruksyon, na tinukoy bilang isang reset ng pangalawang qubit na pinapatakbo ng instruksyon, kasunod ng isang swap ng parehong qubit. Ang operasyong ito ay katumbas ng paglipat ng estado ng unang qubit papunta sa pangalawang qubit, habang sabay na itinatatapon ang papasok na estado ng pangalawang qubit.

Ang package ay idinisenyo upang maging pare-pareho sa paraan ng pagtrato mo sa mga wire cut kapag gumagamit ng mga pisikal na qubit. Halimbawa, ang isang wire cut ay maaaring kumuha ng estado ng pisikal na qubit $n$ at ipagpatuloy ito bilang pisikal na qubit $m$ pagkatapos ng cut. Maaari mong isipin ang "instruction cutting" bilang isang pinag-isang framework para sa pagsasaalang-alang ng parehong wire at gate cut sa loob ng parehong pormalismo (dahil ang isang wire cut ay isang cut [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) na instruksyon lamang). Ang paggamit ng framework na ito para sa wire cutting ay nagbibigay-daan din sa muling paggamit ng qubit, na ipinaliwanag sa seksyon tungkol sa [manu-manong pagputol ng wire](#cut-wires-using-the-low-level-move-instruction).

Ang single-qubit [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) na instruksyon ay nagsisilbing mas abstraktong, mas simpleng interface para sa pagtatrabaho sa mga wire cut. Pinapayagan ka nitong tukuyin kung saan sa circuit dapat putulin ang isang wire sa mataas na antas at hayaan ang circuit cutting addon na mag-insert ng angkop na mga [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) na instruksyon para sa iyo.

Ipinapakita ng sumusunod na halimbawa ang expectation value reconstruction pagkatapos ng wire cutting. Gagawa ka ng isang circuit na may ilang non-local na gate at magtutukoy ng mga observable para matantiya.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit_ibm_runtime import SamplerV2, Batch
from qiskit_aer.primitives import EstimatorV2

from qiskit_addon_cutting.instructions import Move, CutWire
from qiskit_addon_cutting import (
    partition_problem,
    generate_cutting_experiments,
    cut_wires,
    expand_observables,
    reconstruct_expectation_values,
)


qc_0 = QuantumCircuit(7)
for i in range(7):
    qc_0.rx(np.pi / 4, i)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)
qc_0.cx(3, 4)
qc_0.cx(3, 5)
qc_0.cx(3, 6)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)

# Define observable
observable = SparsePauliOp(["ZIIIIII", "IIIZIII", "IIIIIIZ"])

# Draw circuit
qc_0.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg)

## Putulin ang mga wire gamit ang mataas na antas na `CutWire` na instruksyon
Sunod, gumawa ng mga wire cut gamit ang single-qubit [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) na instruksyon sa qubit $q_3$. Kapag handa na ang mga subexperiment para isagawa, gamitin ang [`cut_wires()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#cut_wires) na function para gawing [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) na instruksyon ang `CutWire` sa mga bagong inilalaang qubit.

In [2]:
qc_1 = QuantumCircuit(7)
for i in range(7):
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(CutWire(), [3])
qc_1.cx(3, 4)
qc_1.cx(3, 5)
qc_1.cx(3, 6)
qc_1.append(CutWire(), [3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg)

> **Info:** Kapag ang isang circuit ay pinalawak sa pamamagitan ng isa o higit pang wire cut, ang observable ay kailangang i-update upang masulit ang mga karagdagang qubit na idinagdag. Ang `qiskit-addon-cutting` package ay may convenience function na [`expand_observables()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#expand_observables), na tumatanggap ng mga `PauliList` na object at ng mga orihinal at pinalawak na circuit bilang mga argumento, at nagbabalik ng bagong `PauliList`.
> 
>     Ang ibinalik na `PauliList` na ito ay hindi maglalaman ng anumang impormasyon tungkol sa mga coefficient ng orihinal na observable, ngunit maaari itong balewalain hanggang sa reconstruction ng panghuling expectation value.

In [3]:
# Transform CutWire instructions to Move instructions
qc_2 = cut_wires(qc_1)

# Expand the observable to match the new circuit size
expanded_observable = expand_observables(observable.paulis, qc_0, qc_2)
print(f"Expanded Observable: {expanded_observable}")
qc_2.draw("mpl")

Expanded Observable: ['ZIIIIIIII', 'IIIZIIIII', 'IIIIIIIIZ']


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d398b397-0167-4db9-97ae-6ea502dc43e3-1.svg" alt="Output of the previous code cell" />

### Partition the circuit and observable

Now the problem can be separated into partitions. This is accomplished using the [`partition_problem()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#partition_problem) function with an optional set of partition labels to specify how to separate the circuit. Qubits sharing a common partition label are grouped together, and any non-local gates spanning more than one partition are cut.

If no partition labels are provided, then the partitioning will be automatically determined based on the connectivity of the circuit. Read the next section on [cutting wires manually](#cut-wires-using-the-low-level-move-instruction) for more information on including partition labels.

In [4]:
partitioned_problem = partition_problem(
    circuit=qc_2,
    observables=expanded_observable,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits[0].draw("mpl")

Subobservables to measure: 
{0: PauliList(['IIIII', 'ZIIII', 'IIIIZ']), 1: PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/5fb034f2-da8a-4f4d-ab9b-c990593e04fc-1.svg" alt="Output of the previous code cell" />

In [5]:
subcircuits[1].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg" alt="Output of the previous code cell" />

In this partitioning scheme, you have cut two wires, resulting in a sampling overhead of $4^4$.

### Generate subexperiments to execute and post-process results

To estimate the expectation value of the full-sized circuit, several subexperiments are generated from the decomposed gates' joint quasi-probability distribution and then executed on one (or more) QPUs. The [`generate_cutting_experiments`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) method does this by ingesting arguments for the `subcircuits` and `subobservables` dictionaries you created above, as well as for the number of samples to take from the distribution.

<Admonition type="note" title="Note about the number of samples">
    The `num_samples` argument specifies how many samples to draw from the quasi-probability distribution and determines the accuracy of the coefficients used for the reconstruction. Passing infinity (`np.inf`) ensures all coefficients are calculated exactly. Read the API docs on [generating weights](/docs/api/qiskit-addon-cutting/qpd#generate_qpd_weights) and [generating cutting experiments](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) for more information.
</Admonition>

In [6]:
# Generate subexperiments
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits, observables=subobservables, num_samples=np.inf
)

# Set a backend to use and transpile the subexperiments
backend = FakeManilaV2()
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
isa_subexperiments = {
    label: pass_manager.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Submit each partition's subexperiments to the Qiskit Runtime Sampler
# primitive, in a single batch so that the jobs will run back-to-back.
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
    # Retrieve results
    results = {label: job.result() for label, job in jobs.items()}

Lastly, the expectation value of the full circuit can be reconstructed using the [`reconstruct_expectation_values()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values) method.


The code block below reconstructs the results and compares them with the exact expectation value.

In [7]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
# Apply the coefficients of the original observable
reconstructed_expval = np.dot(reconstructed_expval_terms, observable.coeffs)


# Compute the exact expectation value using the `qiskit_aer` package.
estimator = EstimatorV2()
exact_expval = estimator.run([(qc_0, observable)]).result()[0].data.evs
print(
    f"Reconstructed expectation value: {np.real(np.round(reconstructed_expval, 8))}"
)
print(f"Exact expectation value: {np.round(exact_expval, 8)}")
print(
    f"Error in estimation: {np.real(np.round(reconstructed_expval-exact_expval, 8))}"
)
print(
    f"Relative error in estimation: {np.real(np.round((reconstructed_expval-exact_expval) / exact_expval, 8))}"
)

Reconstructed expectation value: 1.45965266
Exact expectation value: 1.59099026
Error in estimation: -0.1313376
Relative error in estimation: -0.08255085


![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg)

Sa partitioning scheme na ito, nagputol ka ng dalawang wire, na nagresulta sa sampling overhead na $4^4$.

### Bumuo ng mga subexperiment para isagawa at i-post-process ang mga resulta

Para matantiya ang expectation value ng full-sized na circuit, ilang subexperiment ang nabubuo mula sa joint quasi-probability distribution ng mga na-decompose na gate at pagkatapos ay isinasagawa sa isa (o higit pa) na QPU. Ginagawa ito ng [`generate_cutting_experiments`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) na paraan sa pamamagitan ng pag-ingest ng mga argumento para sa mga `subcircuits` at `subobservables` na dictionary na ginawa mo sa itaas, pati na rin ang bilang ng mga sample na kukunin mula sa distribution.

> **Note:** Ang `num_samples` na argumento ay nagtutukoy kung ilang sample ang kukunin mula sa quasi-probability distribution at tinutukoy ang katumpakan ng mga coefficient na ginagamit para sa reconstruction. Ang pagpasa ng infinity (`np.inf`) ay tinitiyak na lahat ng coefficient ay eksaktong kinakalkula. Basahin ang API docs tungkol sa [pagbuo ng mga timbang](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qpd#generate_qpd_weights) at [pagbuo ng mga cutting experiment](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) para sa karagdagang impormasyon.

In [8]:
qc_1 = QuantumCircuit(8)
for i in [*range(4), *range(5, 8)]:
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(Move(), [3, 4])
qc_1.cx(4, 5)
qc_1.cx(4, 6)
qc_1.cx(4, 7)
qc_1.append(Move(), [4, 3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

# Expand observable
observable_expanded = SparsePauliOp(["ZIIIIIII", "IIIIZIII", "IIIIIIIZ"])
qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/15461a2c-85a9-4cb2-a632-b9597ccbc4bd-0.svg" alt="Output of the previous code cell" />

Sa huli, ang expectation value ng buong circuit ay maaaring i-reconstruct gamit ang [`reconstruct_expectation_values()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values) na paraan.

Ang code block sa ibaba ay nire-reconstruct ang mga resulta at inihahambing ang mga ito sa eksaktong expectation value.

In [9]:
partitioned_problem = partition_problem(
    circuit=qc_1,
    partition_labels="AAAABBBB",
    observables=observable_expanded.paulis,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits["A"].draw("mpl")

Subobservables to measure: 
{'A': PauliList(['IIII', 'ZIII', 'IIIZ']), 'B': PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/2139745a-bdc3-40bd-bd6f-d26d2a5b5b14-1.svg" alt="Output of the previous code cell" />

In [10]:
subcircuits["B"].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/4aeb3f1f-a55e-49c4-a7bd-837132429ee1-0.svg" alt="Output of the previous code cell" />

> **Caution:** Para tumpak na ma-reconstruct ang expectation value, ang mga coefficient ng orihinal na observable (na iba sa output ng `generate_cutting_experiments()`) ay dapat na ilapat sa output ng reconstruction, dahil ang impormasyong ito ay nawala nang ang mga cutting experiment ay nabuo o nang ang observable ay pinalawak.
> 
>     Karaniwang ang mga coefficient na ito ay maaaring ilapat sa pamamagitan ng `numpy.dot()` gaya ng ipinakita nang mas maaga.
## Putulin ang mga wire gamit ang mababang antas na `Move` na instruksyon
Ang isang limitasyon ng paggamit ng mas mataas na antas na `CutWire` na instruksyon ay hindi nito pinapayagan ang muling paggamit ng qubit. Kung ito ay nais para sa isang cutting experiment, maaari kang mag-manually na mag-place ng mga [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) na instruksyon. Gayunpaman, dahil ang `Move` na instruksyon ay itinatatapon ang estado ng destination qubit, mahalaga na ang qubit na ito ay hindi nagbabahagi ng anumang entanglement sa natitirang bahagi ng sistema. Kung hindi, ang reset na operasyon ay magiging sanhi ng bahagyang pagbagsak ng estado ng circuit pagkatapos ng wire cut.

Ang code block sa ibaba ay nagsasagawa ng isang wire cut sa qubit $q_3$ para sa parehong halimbawang circuit na ipinakita kanina. Ang pagkakaiba dito ay maaari kang muling gumamit ng qubit sa pamamagitan ng pagbabaligtad ng `Move` na operasyon kung saan ginawa ang pangalawang wire cut (gayunpaman, hindi ito palaging posible at depende sa circuit na pinutol).